<a href="https://colab.research.google.com/github/vorhersager/deep-learning-jax/blob/main/Tutorial_3_Nonconvex_Optimization_in_Deep_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 3: Nonconvex Optimization in Deep Learning


Instructor: John Sipple

### Overview

In Deep Learning, objective functions are typically nonconvex, featuring multiple local minima, saddle points, and plateaus where standard convex optimization guarantees do not apply. This tutorial explores the mechanics of how neural networks navigate these complex loss landscapes. By constructing a functional Feedforward Neural Network (MLP) from scratch using JAX, you will manually implement and analyze the core optimization algorithms that drive modern AI. The notebook moves beyond theory, challenging you to build stateless optimizer functions and visually inspect both the resulting high-dimensional loss surfaces and the physical weights of the trained architecture.

### Key Concepts Explored:

*
**Data Pipeline & Model Architecture:** Preprocessing the MNIST dataset by flattening $28 \times 28$ images into 784-dimensional vectors and normalizing pixel values. You will build a pure functional MLP with a 512-unit ReLU hidden layer and a 10-unit output layer.


*
**Baseline & Momentum Optimizers:** Implementing standard Stochastic Gradient Descent (SGD) , followed by SGD with Nesterov Momentum, which calculates the gradient at a "lookahead" position to effectively correct the accumulated velocity vector.


*
**Adaptive Learning Rates (RMSProp):** Coding RMSProp to tackle the "ravine" problem (steep in one direction, flat in another) by adapting the learning rate for each parameter, dividing the gradient by a running average of its recent magnitude.


*
**Adam (Adaptive Moment Estimation):** Combining the benefits of Momentum (first moment) and RMSProp (second moment). You will implement the full algorithm from scratch, including the critical bias correction step to prevent early training steps from being biased toward zero.


*
**3D Loss Landscape Visualization:** Reducing the vast, high-dimensional parameter space into a 2D slice by generating two random, normalized direction vectors. This allows you to plot and visualize the complex 3D surface of the nonconvex loss function.


*
**Network Weight Visualization:** Rendering a representative subset of the trained architecture's internal connections to inspect what the optimizer actually learned. The visualization utilizes color (blue for positive, red for negative) and varying line thickness to map weight magnitude.



### Real-World Application

Understanding the distinct mechanics of these optimizers is crucial for machine learning engineers diagnosing training instability, navigating vanishing/exploding gradients, or designing custom learning rate schedules for highly specialized datasets where default optimizer configurations fail to converge.

In [ ]:
# --- Dependencies ---
import jax
import jax.numpy as jnp
from jax import grad, jit, vmap, random
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D

# Ensure JAX uses 32-bit (standard for DL) or 64-bit (for precision)
jax.config.update("jax_enable_x64", False)

print(f"JAX Version: {jax.__version__}")
print(f"Devices: {jax.devices()}")

##2. Data Pipeline (MNIST)

We use `tensorflow_datasets` to stream the data, but we convert batches to numpy so JAX can consume them. We treat the images as 1D vectors ($28 \times 28 \to 784$).

In [ ]:
def load_mnist(batch_size=128):
    ds_builder = tfds.builder('mnist')
    ds_builder.download_and_prepare()
    train_ds = tfds.as_numpy(ds_builder.as_dataset(split='train', batch_size=-1))
    test_ds  = tfds.as_numpy(ds_builder.as_dataset(split='test', batch_size=-1))

    # Preprocessing
    def process(data):
        images = data['image']
        labels = data['label']

        # Flatten: (N, 28, 28, 1) -> (N, 784)
        images = images.reshape(images.shape[0], -1)
        # Normalize: [0, 255] -> [0, 1]
        images = images.astype(np.float32) / 255.0
        # One-hot encode labels
        labels = np.eye(10)[labels]
        return images, labels

    train_X, train_Y = process(train_ds)
    test_X, test_Y   = process(test_ds)

    # Create simple generator for batches
    num_train = train_X.shape[0]
    num_batches = num_train // batch_size

    return (train_X, train_Y), (test_X, test_Y), num_batches

# Load Data
BATCH_SIZE = 128
(train_X, train_Y), (test_X, test_Y), num_batches = load_mnist(BATCH_SIZE)

print(f"Input Shape: {train_X.shape}")
print(f"Target Shape: {train_Y.shape}")

In [ ]:
# @title View a few examples
def plot_mnist_samples(X, Y, rows=4, cols=8):
    """
    Plots a grid of MNIST images.
    Args:
        X: Flattened images (N, 784)
        Y: One-hot encoded labels (N, 10)
    """
    fig, axes = plt.subplots(rows, cols, figsize=(1.5*cols, 1.5*rows))

    # Randomly select indices
    indices = np.random.choice(len(X), rows*cols, replace=False)

    for i, ax in enumerate(axes.flat):
        idx = indices[i]

        # Reshape flattened vector back to 28x28 image
        img = X[idx].reshape(28, 28)

        # Get label (argmax of one-hot)
        label = np.argmax(Y[idx])

        ax.imshow(img, cmap='gray')
        ax.set_title(f"Label: {label}")
        ax.axis('off')

    plt.tight_layout()
    plt.show()

print("Example MNIST Training Images:")
plot_mnist_samples(train_X, train_Y)

##3. The Model: Multilayer Perceptron

We define a pure functional model.

* **Input:** 784 dimensions

* **Hidden Layer:** 512 units, ReLU activation

* **Output:** 10 units (Logits)

The parameters are stored in a list of dictionaries: `[{'w':.., 'b':..}, {'w':.., 'b':..}]`.

In [ ]:
def init_mlp_params(key):
    k1, k2 = random.split(key)
    # Layer 1: 784 -> 512
    w1 = random.normal(k1, (784, 512)) * jnp.sqrt(2.0/784) # He Initialization
    b1 = jnp.zeros(512)
    # Layer 2: 512 -> 10
    w2 = random.normal(k2, (512, 10)) * jnp.sqrt(2.0/512)
    b2 = jnp.zeros(10)
    return [{'w': w1, 'b': b1}, {'w': w2, 'b': b2}]

def relu(x):
    return jnp.maximum(0, x)

def softmax_cross_entropy(logits, label):
    # LogSumExp trick for numerical stability
    max_logits = jnp.max(logits, axis=-1, keepdims=True)
    stabilized = logits - max_logits
    log_probs = stabilized - jnp.log(jnp.sum(jnp.exp(stabilized), axis=-1, keepdims=True))
    return -jnp.sum(label * log_probs)

def forward(params, x):
    # Layer 1
    z1 = jnp.dot(x, params[0]['w']) + params[0]['b']
    a1 = relu(z1)
    # Layer 2
    logits = jnp.dot(a1, params[1]['w']) + params[1]['b']
    return logits

def loss_fn(params, x, y):
    logits = forward(params, x)
    loss = softmax_cross_entropy(logits, y)
    return jnp.mean(loss) # Average over batch

Here we render the our MNIST architecture

In [ ]:
import matplotlib.pyplot as plt

def draw_mnist_architecture():
    """
    Visualizes the MLP architecture: 784 -> 512 -> 10
    Draws representative nodes to keep the plot readable.
    """
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis('off')

    # Define layer configurations
    # (Actual Size, Number of nodes to draw, Layer Name, X-position)
    layers = [
        (784, 15, "Input\n(784 px)", 0.1),
        (512, 10, "Hidden\n(512, ReLU)", 0.5),
        (10, 10, "Output\n(10 classes)", 0.9)
    ]

    # Helper to calculate node Y-positions centered vertically
    def get_y_coords(num_nodes):
        spacing = 0.8 / max(num_nodes, 1)
        start = 0.5 + (num_nodes - 1) * spacing / 2
        return [start - i * spacing for i in range(num_nodes)]

    # Draw Nodes and Labels
    node_coords = []
    for actual_size, draw_count, name, x_pos in layers:
        y_coords = get_y_coords(draw_count)
        node_coords.append([(x_pos, y) for y in y_coords])

        # Draw Circles
        for y in y_coords:
            circle = plt.Circle((x_pos, y), 0.02, color='skyblue', ec='k', zorder=10)
            ax.add_artist(circle)

        # Add "..." to indicate there are more nodes than shown
        if actual_size > draw_count:
            ax.text(x_pos, y_coords[-1] - 0.1, "...", ha='center', fontsize=14, weight='bold')

        # Layer Label
        ax.text(x_pos, 0.95, name, ha='center', va='bottom', fontsize=12, weight='bold')

    # Draw Edges (Weights)
    # We draw lines from every drawn node in layer i to every drawn node in layer i+1
    for i in range(len(node_coords) - 1):
        source_nodes = node_coords[i]
        target_nodes = node_coords[i+1]

        for sx, sy in source_nodes:
            for tx, ty in target_nodes:
                line = plt.Line2D([sx, tx], [sy, ty], color='gray', alpha=0.2, lw=0.5)
                ax.add_artist(line)

    plt.title("MLP Architecture for MNIST", fontsize=16)
    plt.tight_layout()
    plt.show()

draw_mnist_architecture()

##4. Optimizer Implementations

We will implement the update rules manually. In JAX, optimizers are **stateless functions** that take current parameters and optimizer state (like velocity), and return new parameters and new state.



###4.1 Stochastic Gradient Descent (SGD)

The most basic first-order optimizer.

$$\theta_{t+1} = \theta_t - \eta \nabla_\theta J(\theta_t)$$

* **Pros:** Simple, unbiased estimate of the gradient.
* **Cons:** Slow convergence, oscillates in ravines, gets stuck in saddle points.

In [ ]:
@jit
def update_sgd(params, grads, state, lr=0.01):
    """
    Standard SGD update.
    State: None (SGD is stateless)
    """
    new_params = []
    for p, g in zip(params, grads):
        new_layer = {}
        new_layer['w'] = p['w'] - lr * g['w']
        new_layer['b'] = p['b'] - lr * g['b']
        new_params.append(new_layer)
    return new_params, state

###4.2 SGD with Nesterov Momentum

Momentum accumulates a velocity vector $v$ to dampen oscillations. Nesterov calculates the gradient at the "lookahead" position, effectively correcting the velocity vector.

$$v_{t+1} = \gamma v_t - \eta \nabla J(\theta_t + \gamma v_t)$$$$\theta_{t+1} = \theta_t + v_{t+1}$$

*Note*: In practice (and below), we often apply the standard momentum update but use the Nesterov formulation for the gradient step.

In [ ]:
def init_momentum_state(params):
    # Velocity is same shape as params, initialized to 0
    return [{'w': jnp.zeros_like(p['w']), 'b': jnp.zeros_like(p['b'])} for p in params]

@jit
def update_nesterov(params, grads, state, lr=0.01, gamma=0.9):
    """
    SGD with Nesterov Momentum.
    State: Velocity (v)
    """
    new_params = []
    new_state = []

    for p, g, v in zip(params, grads, state):
        # Update Velocity
        v_w_new = gamma * v['w'] + lr * g['w']
        v_b_new = gamma * v['b'] + lr * g['b']

        # Nesterov Update
        # theta_new = theta - (gamma * v_prev + (1+gamma) * v_new) ?
        # Simplified standard implementation usually just applies the velocity:
        w_new = p['w'] - v_w_new
        b_new = p['b'] - v_b_new

        new_params.append({'w': w_new, 'b': b_new})
        new_state.append({'w': v_w_new, 'b': v_b_new})

    return new_params, new_state

###4.3 RMSProp (Root Mean Square Propagation)

RMSProp adapts the learning rate for each parameter. It divides the gradient by a running average of its recent magnitude. This handles the "ravine" problem (steep in one direction, flat in another).

$$E[g^2]_t = \beta E[g^2]_{t-1} + (1-\beta)g_t^2$$$$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{E[g^2]_t + \epsilon}} g_t$$

In [ ]:
def init_rmsprop_state(params):
    # Cache (Moving average of squared gradients)
    return [{'w': jnp.zeros_like(p['w']), 'b': jnp.zeros_like(p['b'])} for p in params]

@jit
def update_rmsprop(params, grads, state, lr=0.001, beta=0.9, eps=1e-8):
    new_params = []
    new_state = []

    for p, g, s in zip(params, grads, state):
        # Update moving average of squared gradients
        s_w_new = beta * s['w'] + (1 - beta) * jnp.square(g['w'])
        s_b_new = beta * s['b'] + (1 - beta) * jnp.square(g['b'])

        # Parameter Update
        w_new = p['w'] - lr * g['w'] / (jnp.sqrt(s_w_new) + eps)
        b_new = p['b'] - lr * g['b'] / (jnp.sqrt(s_b_new) + eps)

        new_params.append({'w': w_new, 'b': b_new})
        new_state.append({'w': s_w_new, 'b': s_b_new})

    return new_params, new_state

###4.4 Adam (Adaptive Moment Estimation)

Adam combines Momentum (first moment) and RMSProp (second moment). It also includes bias correction to prevent initial steps from being biased toward zero.

1. **First Moment** ($m$): $m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t$
2. **Second Moment** ($v$): $v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2$
3. **Bias Correction**: $\hat{m}_t = m_t / (1-\beta_1^t)$, $\hat{v}_t = v_t / (1-\beta_2^t)$
4. **Update**: $\theta_{t+1} = \theta_t - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$

In [ ]:
def init_adam_state(params):
    # Returns (m, v, t)
    m = [{'w': jnp.zeros_like(p['w']), 'b': jnp.zeros_like(p['b'])} for p in params]
    v = [{'w': jnp.zeros_like(p['w']), 'b': jnp.zeros_like(p['b'])} for p in params]
    t = 0
    return (m, v, t)

@jit
def update_adam(params, grads, state, lr=0.001, b1=0.9, b2=0.999, eps=1e-8):
    m_prev, v_prev, t_prev = state
    t = t_prev + 1
    new_params = []
    new_m = []
    new_v = []

    for p, g, m, v in zip(params, grads, m_prev, v_prev):
        # 1. Update First Moment (Momentum)
        m_w = b1 * m['w'] + (1 - b1) * g['w']
        m_b = b1 * m['b'] + (1 - b1) * g['b']

        # 2. Update Second Moment (RMSProp)
        v_w = b2 * v['w'] + (1 - b2) * jnp.square(g['w'])
        v_b = b2 * v['b'] + (1 - b2) * jnp.square(g['b'])

        # 3. Bias Correction
        m_w_hat = m_w / (1 - b1 ** t)
        m_b_hat = m_b / (1 - b1 ** t)
        v_w_hat = v_w / (1 - b2 ** t)
        v_b_hat = v_b / (1 - b2 ** t)

        # 4. Update Parameters
        w_new = p['w'] - lr * m_w_hat / (jnp.sqrt(v_w_hat) + eps)
        b_new = p['b'] - lr * m_b_hat / (jnp.sqrt(v_b_hat) + eps)

        new_params.append({'w': w_new, 'b': b_new})
        new_m.append({'w': m_w, 'b': m_b})
        new_v.append({'w': v_w, 'b': v_b})

    return new_params, (new_m, new_v, t)

##5. Training Loop & Comparison

We define a generic training loop that accepts an update function and runs for one epoch (to keep this tutorial brief).

In [ ]:
def train_optimizer(opt_name, update_fn, init_state_fn, params, lr=0.001):
    print(f"Training {opt_name}...")

    # Initialize Optimizer State
    if init_state_fn:
        opt_state = init_state_fn(params)
    else:
        opt_state = None

    losses = []

    # Define the gradient function
    grad_fn = jax.grad(loss_fn)

    # Batch Loop
    # We simply iterate through the numpy arrays
    for i in range(num_batches):
        start = i * BATCH_SIZE
        end = start + BATCH_SIZE
        x_batch = train_X[start:end]
        y_batch = train_Y[start:end]

        # Compute Gradients
        grads = grad_fn(params, x_batch, y_batch)

        # Update
        params, opt_state = update_fn(params, grads, opt_state, lr=lr)

        if i % 20 == 0:
            l = loss_fn(params, x_batch, y_batch)
            losses.append(float(l))

    return losses, params

# Initialize generic params
start_params = init_mlp_params(random.PRNGKey(99))

# Run Comparisons
# Note: Different optimizers often need different LRs to be fair,
# but we will use standard defaults to show behavior.

loss_sgd, p_sgd = train_optimizer("SGD", update_sgd, None, start_params, lr=0.1)
loss_nes, p_nes = train_optimizer("Nesterov", update_nesterov, init_momentum_state, start_params, lr=0.01)
loss_rms, p_rms = train_optimizer("RMSProp", update_rmsprop, init_rmsprop_state, start_params, lr=0.001)
loss_adm, p_adm = train_optimizer("Adam", update_adam, init_adam_state, start_params, lr=0.001)

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(loss_sgd, label='SGD (lr=0.1)')
plt.plot(loss_nes, label='Nesterov (lr=0.01)')
plt.plot(loss_rms, label='RMSProp (lr=0.001)')
plt.plot(loss_adm, label='Adam (lr=0.001)')
plt.yscale('log')
plt.xlabel("Iterations (x20)")
plt.ylabel("Cross Entropy Loss")
plt.title("Optimizer Convergence Comparison")
plt.legend()
plt.grid(True)
plt.show()

##6. Visualizing the Loss Landscape (3D)

Visualizing the loss surface of a neural network ($\mathbb{R}^{1000s} \to \mathbb{R}$) is impossible directly. We use a reduction technique (Li et al., 2018):

1. Take the trained parameters $\theta^*$.
2. Choose two random direction vectors $\delta$ and $\eta$ (normalized).
3. Plot $f(\alpha, \beta) = L(\theta^* + \alpha \delta + \beta \eta)$.

This creates a 2D slice through the high-dimensional nonconvex surface.

In [ ]:
def get_random_direction(key, params):
    # Creates a random direction vector with same shape as params
    direction = []
    keys = random.split(key, len(params))
    for i, p in enumerate(params):
        k_w, k_b = random.split(keys[i])
        d_w = random.normal(k_w, p['w'].shape)
        d_b = random.normal(k_b, p['b'].shape)

        # Filter Normalization (Normalize by weight magnitude to keep scale consistent)
        d_w = d_w * (jnp.linalg.norm(p['w']) / (jnp.linalg.norm(d_w) + 1e-10))
        d_b = d_b * (jnp.linalg.norm(p['b']) / (jnp.linalg.norm(d_b) + 1e-10))

        direction.append({'w': d_w, 'b': d_b})
    return direction

def perturb_params(params, alpha, dir1, beta, dir2):
    # new_p = p + alpha*d1 + beta*d2
    new_params = []
    for p, d1, d2 in zip(params, dir1, dir2):
        nw = p['w'] + alpha * d1['w'] + beta * d2['w']
        nb = p['b'] + alpha * d1['b'] + beta * d2['b']
        new_params.append({'w': nw, 'b': nb})
    return new_params

# 1. Setup Directions based on Adam's result

def render_losslandscape(p_opt, opt_name, seed = 7511):
  key = random.PRNGKey(seed)
  k1, k2 = random.split(key)
  dir1 = get_random_direction(k1, p_opt)
  dir2 = get_random_direction(k2, p_opt)

  # 2. Create Grid
  resolution = 20
  range_val = 1.0
  alphas = np.linspace(-range_val, range_val, resolution)
  betas = np.linspace(-range_val, range_val, resolution)
  X, Y = np.meshgrid(alphas, betas)
  Z = np.zeros_like(X)

  # 3. Compute Loss on Grid (using a fixed batch for speed)
  fixed_x = train_X[:1024]
  fixed_y = train_Y[:1024]

  print("Computing Loss Landscape (this may take a moment)...")
  for i in range(resolution):
      for j in range(resolution):
          # Perturb
          p_temp = perturb_params(p_opt, X[i,j], dir1, Y[i,j], dir2)
          # Calc Loss
          loss_val = loss_fn(p_temp, fixed_x, fixed_y)
          Z[i,j] = float(loss_val)

  # 4. Plot 3D
  fig = plt.figure(figsize=(12, 8))
  ax = fig.add_subplot(111, projection='3d')
  surf = ax.plot_surface(X, Y, Z, cmap='viridis', edgecolor='none', alpha=0.9)
  ax.set_title(f"Loss Landscape near {opt_name} Optima (2D Slice)")
  ax.set_xlabel("Direction 1 (alpha)")
  ax.set_ylabel("Direction 2 (beta)")
  ax.set_zlabel("Loss")
  fig.colorbar(surf, shrink=0.5, aspect=5)
  plt.show()

seed = 111
render_losslandscape(p_sgd, 'SGD', seed)
render_losslandscape(p_nes, 'SGD + Nesterov', seed)
render_losslandscape(p_rms, 'RMSProp', seed)
render_losslandscape(p_adm, 'Adam', seed)

## 7. Visualizing the trained architecture

Because there are over 400,000 weights, we cannot draw them all. This visualization renders a **representative subset** of connections:

1. We select representative neurons from each layer (e.g., 20 out of 784 inputs).

2. We draw the specific connections between these selected neurons based on the trained parameter matrices.

3. **Color**: Blue = Positive weight, Red = Negative weight.

4. **Thickness**: Proportional to the magnitude (absolute value) of the weight.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import jax.numpy as jnp # Ensure jnp is imported if not already in context

def draw_trained_architecture(params, opt_name):
    """
    Visualizes a subset of learned weights from the trained MLP.
    Args:
        params: The list of parameter dictionaries [{'w':.., 'b':..}, ...]
    """
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.axis('off')

    # --- 1. Configuration & Node Placement ---
    # Define actual dimensions and how many nodes to draw representatively
    # (Actual Dim, Drawn Nodes, Label, X-position)
    layers_config = [
        (784, 20, "Input (784)\nSubset shown", 0.1),
        (512, 15, "Hidden (512)\nSubset shown", 0.5),
        (10, 10,  "Output (10)\nAll shown", 0.9)
    ]

    # Helper to calculate vertical positions for nodes
    def get_y_coords(num_draw, actual_dim):
        # Spread nodes out, centered vertically
        spacing = 0.85 / max(num_draw, 1)
        start = 0.5 + (num_draw - 1) * spacing / 2
        coords = [start - i * spacing for i in range(num_draw)]

        # Determine which actual indices these drawn nodes represent
        indices = np.linspace(0, actual_dim-1, num_draw, dtype=int)
        return coords, indices

    node_positions = []
    node_indices = []

    # Calculate positions and indices for all layers
    for actual, drawn, label, xpos in layers_config:
        y_coords, idxs = get_y_coords(drawn, actual)
        node_positions.append([(xpos, y) for y in y_coords])
        node_indices.append(idxs)
        ax.text(xpos, 0.98, label, ha='center', va='top', fontsize=11, weight='bold')

    # --- 2. Weight Visualization (Edges) ---

    # Find global max magnitude to normalize thickness
    max_w_mag = 0.0
    for layer_p in params:
        # jnp.max returns a JAX array, we cast to float
        layer_max = float(jnp.max(jnp.abs(layer_p['w'])))
        max_w_mag = max(max_w_mag, layer_max)

    # Avoid division by zero
    max_w_mag = max_w_mag if max_w_mag > 1e-9 else 1.0

    # Iterate through layers to draw connections
    for l in range(len(params)):
        src_coords = node_positions[l]
        src_idxs = node_indices[l]
        tgt_coords = node_positions[l+1]
        tgt_idxs = node_indices[l+1]

        W_actual = params[l]['w']

        for i, (sx, sy) in enumerate(src_coords):
            real_src_idx = src_idxs[i]
            for j, (tx, ty) in enumerate(tgt_coords):
                real_tgt_idx = tgt_idxs[j]

                # Get weight value and cast to standard python float
                w_val = float(W_actual[real_src_idx, real_tgt_idx])

                # Determine Style
                color = '#1f77b4' if w_val > 0 else '#d62728'

                magnitude = abs(w_val)

                # Calculate thickness and alpha, ensuring they are floats
                thickness = float(0.1 + 3.9 * (magnitude / max_w_mag))
                alpha_val = float(0.3 + 0.7 * (magnitude / max_w_mag))

                # Clip alpha to [0, 1] just in case
                alpha_val = min(max(alpha_val, 0.0), 1.0)

                # Draw line
                line = plt.Line2D([sx, tx], [sy, ty], color=color, lw=thickness, alpha=alpha_val, zorder=1)
                ax.add_artist(line)

    # --- 3. Draw Nodes (on top of edges) ---
    for l_coords in node_positions:
        for nx, ny in l_coords:
            circle = plt.Circle((nx, ny), 0.015, color='white', ec='k', lw=1.5, zorder=10)
            ax.add_artist(circle)

    plt.title(f"Visualization of Learned Weights ({opt_name})\nBlue (+), Red (-), Thickness denotes Magnitude", fontsize=14)
    plt.tight_layout()
    plt.show()

# Visualize the parameters learned by the Adam optimizer
start_params = init_mlp_params(random.PRNGKey(99))

draw_trained_architecture(start_params, 'Start Params')
draw_trained_architecture(p_sgd, 'SGD Optimizer')
draw_trained_architecture(p_nes, 'SGD + Nesterov Optimizer')
draw_trained_architecture(p_rms, 'RMSProp Optimizer')
draw_trained_architecture(p_adm, 'Adam Optimizer')